In [ ]:
import nest_asyncio
nest_asyncio.apply()


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
torch.cuda.empty_cache()

In [ ]:
#pip install llama-index==0.10.30

In [ ]:
import torch

#!pip uninstall -y keras keras-nlp
!pip install -U transformers accelerate bitsandbytes
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-huggingface
!pip install llama-index-vector-stores-chroma
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf
!pip install huggingface_hub


In [ ]:
#!pip install tensorflow # tensorflow to resolve backend error
#!pip install tensorflow-text # Added to resolve tensorflow_text backend error
!pip install llama-index-retrievers-bm25 # Added for BM25Retriever

In [ ]:
#unnecessary, redundant
#from huggingface_hub import login
#login()

In [ ]:
#Hugging Faqce Login
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN')) #HF_TOKEN is set in colab secrets


In [ ]:
#Upload pdf files
from google.colab import files
uploaded = files.upload()

In [ ]:
import os

os.makedirs("data", exist_ok=True)

for file in uploaded.keys():
    os.rename(file, f"data/{file}")

In [ ]:
'''
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("data").load_data()

print(len(documents))
'''

**Approach 1**
#Manual Parsing + conversion

In [ ]:
'''
import os
from llama_parse import LlamaParse
from llama_index.core import Document as LlamaIndexDocument

# 1. Parse PDFs with LlamaParse
parser = LlamaParse(
    api_key="llx-LqND1MZRbdpBmKSv7ENsm0beovI9z3Ov8Bf23yY6UBo2fXOY", # or set env var LLAMA_CLOUD_API_KEY
    result_type="markdown",
    verbose=True,
    num_workers=4
)

pdf_files = [os.path.join("./data", f) for f in os.listdir("./data") if f.endswith(".pdf")]
parsed_results = parser.load_data(pdf_files)   # returns list of lists of llama_parse Documents

documents = []
for file_result in parsed_results:
    for doc in file_result:
        llama_index_doc = LlamaIndexDocument(
            text=doc.text,
            metadata=doc.metadata
        )
        documents.append(llama_index_doc)
'''

**Approach 2**
 #integration via SimpleDirectoryReader and file_extractor

In [ ]:
!pip install llama-parse

In [ ]:
!pip install -U llama-index-readers-file


In [ ]:
from llama_parse import LlamaParse
from llama_index.core import Document as LlamaIndexDocument, SimpleDirectoryReader

# 1. Parse PDFs with LlamaParse

Llama_parser_key=userdata.get('LLAMA_CLOUD_API_KEY')

parser = LlamaParse(api_key=Llama_parser_key,
    result_type="markdown",
    verbose=True,
    num_workers=4)

file_extractor = {".pdf": parser}

documents = SimpleDirectoryReader("./data",
    file_extractor=file_extractor).load_data()


In [ ]:
print(len(documents))
#This will show the parsed markdown text returned by LlamaParse.

In [ ]:
#Print the Parsed Content
#This will show the parsed markdown text returned by LlamaParse.
print(documents[0].text)

In [ ]:
#Print Metadata
#Each document also has metadata.
print(documents[0].metadata)

In [ ]:
print("Total parsed documents:", len(documents))

for i, doc in enumerate(documents):
    print("\n-------------")
    print(f"Document {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.text[:500])   # first 500 characters

In [ ]:
#Chunk docs (chunking)

from llama_index.core.node_parser import MarkdownNodeParser

node_parser = MarkdownNodeParser(chunk_size=512,chunk_overlap=50)
nodes = node_parser.get_nodes_from_documents(documents)

In [ ]:
'''
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2")
'''

In [ ]:
#only created an object: embed_model
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5",device="cpu")

print("Embedding model loaded successfully")

In [ ]:
#Embeddings are applied during index creation.
#In LlamaIndex, the embedding happens when building the index.

In [ ]:
#quantized version
'''
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm = HuggingFaceLLM(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    tokenizer_name="meta-llama/Llama-2-7b-chat-hf",
    context_window=4096,
    max_new_tokens=512, #increased for better summary
    generate_kwargs={"temperature":0.7},
    device_map="auto",
    model_kwargs={"quantization_config": quant_config},
)
'''

In [ ]:
import torch

from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import BitsAndBytesConfig

# 1. Improved Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, # Use torch.float16 instead of "float16" string
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# 2. Updated LLM Call
llm = HuggingFaceLLM(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    tokenizer_name="meta-llama/Llama-2-7b-chat-hf",
    context_window=4096,
    max_new_tokens=512,#increased for better summary
    generate_kwargs={"temperature": 0.7, "do_sample": True}, # Added do_sample for temperature to work
    device_map="auto",
    model_kwargs={"quantization_config": quant_config},
)

In [ ]:
'''
#storing chroma vector DB (skip)
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# Create a client and a collection
db_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db_client.get_or_create_collection("my_rag_collection")

# Assign it to LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
'''

In [ ]:
import torch
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
#from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Global Configuration (The Brain and Eyes) global settings
# Using Settings makes code much cleaner and prevents "NameErrors"
Settings.llm = llm # Using quantized Llama-2 which is already loaded
Settings.embed_model = embed_model #HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
## These below settings are not used because we already chunked manually
Settings.chunk_size = 512 # Smaller chunks are better for RAG accuracy
Settings.chunk_overlap = 50   # recommended for better context continuity

# 2. Setup Persistent ChromaDB (The "Memory")
# This saves your vector data to a folder so it survives a crash/restart
# Ensure a clean start for ChromaDB
db = chromadb.PersistentClient(path="./chroma_db") #a db_client  a client
chroma_collection = db.get_or_create_collection("pdf_rag_collection") #collection

# 3. Connect Chroma to LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

'''
4. Load and Index Documents
Make sure you have a folder named 'data' with your PDFs in
documents = SimpleDirectoryReader("./data").load_data()
'''
# This line creates the index and automatically stores it in Chroma
#Build Index from Nodes
index=VectorStoreIndex(nodes,# nodes is the list from MarkdownNodeParser
                       storage_context=storage_context,
                       show_progress=True)

'''
index = VectorStoreIndex.from_documents(
    nodes,
    storage_context=storage_context,
    show_progress=True) # Lets you see the embedding progress
    #Now embeddings will be created for your parsed nodes. (nodes instead of documents)
'''
# 5. Create the Query Engine
#query_engine = index.as_query_engine(streaming=True) # Streaming makes it feel faster

print("\n RAG Pipeline Ready!")

# #2 Retrieval Pipeline
Everything below happens after the index is built.
This is where hybrid search and reranking live.

In [ ]:
!pip install FlagEmbedding
!pip install llama-index-postprocessor-flag-embedding-reranker


In [ ]:
pip install llama-index-retrievers-bm25

In [ ]:
#Hybrid Retriever block
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.retrievers.bm25 import BM25Retriever

#step 1 vector retriever- semantic (Uses embeddings stored in Chroma.)
vector_retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=4) #this performs semantic search

#step 2 BM25 retrievr- keyword (this works directly on text nodes )
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=4) #this performs keyword matching

#step 3 combine them: Hybrid fusion  (hybrid) now merge both retrievers
from llama_index.core.retrievers import QueryFusionRetriever

hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=4,
    num_queries=1,
    mode="reciprocal_rerank"
)

#now #Reranker (reranking implementation)
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker

reranker = FlagEmbeddingReranker(
    model="BAAI/bge-reranker-base", #reranker
    top_n=3)


In [ ]:
# Query Rewriting (HyDE Transform)
from llama_index.core.query_engine import RetrieverQueryEngine

# HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
# Correct HyDE import path
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

# Create HyDE transformer
hyde = HyDEQueryTransform(include_original=True)

# Wrap hybrid retriever with HyDE
query_engine = TransformQueryEngine(
    query_engine=RetrieverQueryEngine.from_args(
        retriever=hybrid_retriever,
        node_postprocessors=[reranker],
        streaming=False),
    query_transform=hyde)

In [ ]:
llm_answer=llm.complete("what are the steps in RAGAS evaluation?")
print(llm_answer)

In [ ]:

#ask question and get the aswer

question="Give me a detailed summary of the main points in these documents."
print(f"\n{'='*50}")
print(f"question: {question}")
print(f"{'='*50}\n")

response = query_engine.query(question)

# Print Answer
print("\n📌 Answer:\n")
print(response.response)

# Print Citations (Sources)
print("\n📚 Sources:\n")

for i, source_node in enumerate(response.source_nodes):
    node = source_node.node
    metadata = node.metadata if hasattr(node, "metadata") else {}
    text_preview = node.get_text()[:200]

    print(f"[Source {i+1}]")
    print(f"Metadata: {metadata}")
    print(f"Preview: {text_preview}")
    print("-" * 50)

manual eval = ground truth sanity check,

(LLM evaluating LLM-generated data)
synthetic eval = scale testing
#Manual Evaluation

In [ ]:
!pip install ragas datasets langchain-groq sentence-transformers

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
'''
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper

llm_groq = ChatGroq(
    model_name="mixtral-8x7b-32768",
    temperature=0
)

#initializing evaluator LLM, seperate from generation llm model
evaluator_llm = LangchainLLMWrapper(llm_groq)
'''

Evaluator LLM ready using (Groq + Mixtral)

In [ ]:
'''
questions = [
    "What is TCS Q2 revenue?",
    "Summarize the document",
    "What are key highlights?"
]

data = []

for q in questions:
    response = query_engine.query(q)

    answer = str(response)

    # FIXED extraction
    contexts = [node.node.text for node in response.source_nodes]

    data.append({
        "question": q,
        "answer": answer,
        "contexts": contexts
    })
'''

In [ ]:
#rom datasets import Dataset

#dataset = Dataset.from_list(data)

In [ ]:
'''
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

result = evaluate(dataset,
    metrics=[faithfulness,
        answer_relevancy,
        context_precision,
        context_recall],
    llm=evaluator_llm)
'''

In [ ]:
#print(result)
'''
print(result)
print("\nAverage scores:")
import numpy as np
for metric in metrics:
  print(f"{metric.name}: {np.mean(result[metric.name]):.3f}")
'''

#Synthetic evaluation (Automatic)

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [ ]:
#import os
#from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
!pip install rapidfuzz

In [ ]:
import langchain
import ragas
import pydantic

print(langchain.__version__)
print(ragas.__version__)
print(pydantic.__version__)

In [ ]:
import os
from datasets import Dataset

from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from llama_index.core.node_parser import SentenceSplitter

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.persona import Persona

try:
    from ragas.testset.transforms.extractors import NERExtractor, EmbeddingExtractor
except ImportError:
    from ragas.testset.transforms.extractors.llm_based import NERExtractor
    from ragas.testset.transforms.extractors.embeddings import EmbeddingExtractor


print("Initializing Models...")
groq_chat = ChatGroq(model="llama-3.1-8b-instant",
    temperature=0.0,
    max_tokens=1024) # reduced to avoid the 6000 TPM limit

hf_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'})

generator_llm = LangchainLLMWrapper(groq_chat)
generator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

print("Chunking Documents...")
splitter = SentenceSplitter(chunk_size=1000, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)

print("Building Knowledge Graph...")
kg = KnowledgeGraph()
# 1 chunk to prevent Ragas from firing concurrent requests
for i, node in enumerate(nodes[:1]):
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": str(node.text),
                "document_metadata": {"source": f"chunk_{i}"}
            }
        )
    )

print("Applying Transforms...")
embedding_extractor = EmbeddingExtractor(embedding_model=generator_embeddings)
ner_extractor = NERExtractor(llm=generator_llm)
apply_transforms(kg, transforms=[embedding_extractor, ner_extractor])

print("Configuring Synthesizers...")
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0)
]

print("Defining Personas...")
personas = [
    Persona(
        name="General User",
        role_description="A standard user asking factual questions based exactly on the provided text."
    )
]

print("Generating Testset...")
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=kg,
    persona_list=personas
)

# Exactly ONE question. We just need it to succeed once.
testset = generator.generate(testset_size=1, query_distribution=query_distribution)
# 1. Convert to Pandas (This usually works in 0.4.x)
test_df = testset.to_pandas()

# 2. Convert that DataFrame to a HuggingFace Dataset for Ragas Evaluate
test_dataset = Dataset.from_pandas(test_df)

print("Testset generation complete! ")
print(test_df.head())

In [ ]:
# Convert to Hugging Face Dataset (contains question, reference, contexts)
#test_dataset = testset.to_dataset()

In [ ]:
'''
# 4. Now run your RAG pipeline on each generated question
eval_samples = []
for item in test_dataset:
    question = item["question"]
    # Query your engine (assumes streaming=False, so response.response is the answer)
    response = query_engine.query(question)
    answer = response.response if hasattr(response, "response") else str(response)
    # Retrieved contexts from your retriever (available because streaming=False)
    contexts = [node.node.text for node in response.source_nodes]   # adjust if needed
    # Append with reference from generated set
    eval_samples.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "reference": item["reference"]
    })

eval_dataset = Dataset.from_list(eval_samples)
'''

# upgraded mapping code

In [ ]:
from ragas import EvaluationDataset

# RAG pipeline on each generated question
eval_samples = []

print("Querying RAG engine...")
for item in test_dataset:
    #changed 'question' to 'user_input'
    query = item["user_input"]

    # Query your engine (assumes streaming=False)
    response = query_engine.query(query)
    answer = response.response if hasattr(response, "response") else str(response)

    # Retrieved contexts from your retriever
    contexts = [node.node.text for node in response.source_nodes]

    # naming conventions for the evaluation dict
    eval_samples.append({
        "user_input": query,                 # Formerly 'question'
        "response": answer,                  # Formerly 'answer'
        "retrieved_contexts": contexts,      # Formerly 'contexts'
        "reference": item["reference"]       # Formerly 'ground_truth'
    })

# evaluate() strictly requires an EvaluationDataset object
eval_dataset = EvaluationDataset.from_list(eval_samples)

print(f"Evaluation dataset ready! Prepared {len(eval_samples)} sample(s).")

# The Final Step: The Evaluation
To make sure you get this project completely finished today without another hiccup, here is the upgraded final block.

We wrap the evaluator LLM exactly the way we wrapped the generator LLM, and pass it directly into the evaluate() function.

In [ ]:
import warnings
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 1. THE FIX: Forcefully mute Ragas's broken DeprecationWarnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 2. THE FIX: Revert to the legacy metric imports which evaluate() actually accepts
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

print("1. Querying your RAG engine...")
eval_samples = []

for item in test_dataset:
    query = item["user_input"]

    # Query your LlamaIndex engine
    response = query_engine.query(query)
    answer = response.response if hasattr(response, "response") else str(response)
    contexts = [node.node.text for node in response.source_nodes]

    # Map to strict Ragas 0.4 dictionary format
    eval_samples.append({
        "user_input": query,
        "response": answer,
        "retrieved_contexts": contexts,
        "reference": item["reference"]
    })

eval_dataset = EvaluationDataset.from_list(eval_samples)

print("2. Running Evaluation Metrics...")

# Wrap the models you already have loaded in memory
ragas_eval_llm = LangchainLLMWrapper(groq_chat)
ragas_eval_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_eval_llm,
    embeddings=ragas_eval_embeddings
)

print("\n- FINAL EVALUATION RESULTS -")
print(result)

In [ ]:
print(eval_dataset.to_pandas()[["user_input", "response", "reference"]])

In [ ]:
df_result = result.to_pandas()
df_result.to_csv("baseline_evaluation.csv", index=False)

In [ ]:
# evaluator llm is for scoring - (strict judge eval ie temp 0)
#(ques+ answers+context)= scoring
# 5. Evaluate using the same evaluator LLM (or a separate one)
#from ragas.llms import llm_factory # Added import for llm_factory
#from langchain_groq import ChatGroq # Ensure ChatGroq is imported here if not globally available

'''
evaluator_llm = ChatGroq(model_name="mixtral-8x7b-32768", temperature=0)

result = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm
)
print("done")
print(result)
'''

In [ ]:
#2nd question
response = query_engine.query("What did this financial document say about TCS stock? list all the sayings in brief.")
print(response)

In [ ]:

# The simplest way to ask a question
question = "Give me a detailed summary of the main points in these documents."

response = query_engine.query(question)

print("-" * 30)
print(f"QUESTION: {question}")
print("-" * 30)
print(f"ANSWER: {response}")
print("-" * 30)


In [ ]:
print(llm.complete("hie man say something in bengali language'"))
#not rag, just LLM brain

In [ ]:
import sys

print("--- Hybrid RAG Chatbot Loaded (type 'exit' to quit) ---")

while True:
    user_input = input("\nUser: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input.strip():
        continue

    print("Assistant: ", end="", flush=True)

    # Generate the non-streaming response
    response = query_engine.query(user_input)

    # Print the complete response directly
    print(response.response)
    print() # New line after the full response


In [ ]:
print(chroma_collection.count())

In [ ]:
#delete the database folder
import shutil
shutil.rmtree("./chroma_db")

In [ ]:
#duplicate embeddings accumulate (need this before creating client)
import shutil
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")